# Fig. 2 U64 Sampler Check: DDPM500 vs DPM-Solver50

This notebook inspects the sampler comparison for the long-trained Fig. 2 run `nf_fig2_u64_d2p15_noaug_200k`.

- DDPM reference label: `ddpm500_wrapper`, generated with `DDPMScheduler` and `num_steps=500` through `scripts/sample_cosmodiff.py`.
- DPM test label: `dpm50`, generated with `DPMSolverMultistepScheduler` and `num_steps=50` through the same wrapper.
- Run only: Fig. 2 `u64`, `d2p15`, `dataset_size=32768`, latest checkpoint `checkpoint-epoch-0586` on Great Lakes.

Run from the repo root on Great Lakes after both sampler jobs have completed.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'scripts').exists()), Path.cwd()).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.metrics import batch_power_spectra

SWEEP_NAME = os.environ.get('NF_SAMPLER_SWEEP_NAME', 'nf_generalize_fig2')
RUN_NAME = os.environ.get('NF_SAMPLER_RUN_NAME', 'nf_fig2_u64_d2p15_noaug_200k')
SEED = int(os.environ.get('NF_SAMPLER_SEED', '123'))
DDPM_LABEL = os.environ.get('NF_SAMPLER_DDPM_LABEL', 'ddpm500_wrapper')
DPM_LABEL = os.environ.get('NF_SAMPLER_DPM_LABEL', 'dpm50')
SAMPLER_LABELS = {
    'raw_train_full': 'DDPM500 original',
    'ddpm500_wrapper': 'DDPM500 wrapper',
    'dpm50': 'DPM-Solver50',
}
DDPM_NAME = SAMPLER_LABELS.get(DDPM_LABEL, DDPM_LABEL)
DPM_NAME = SAMPLER_LABELS.get(DPM_LABEL, DPM_LABEL)
MAX_SAMPLES = int(os.environ.get('NF_SAMPLER_MAX_SAMPLES', '512'))

SAMPLE_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'sampler_compare'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'

DDPM_PATH = SAMPLE_DIR / f'{RUN_NAME}_seed{SEED}_{DDPM_LABEL}.npz'
DPM_PATH = SAMPLE_DIR / f'{RUN_NAME}_seed{SEED}_{DPM_LABEL}.npz'
CSV_PATH = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_vs_{DPM_LABEL}_sampler_compare.csv'
SUMMARY_PATH = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_vs_{DPM_LABEL}_sampler_compare_summary.json'

RUN_ROW = {}
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    RUN_ROW = next((row for row in manifest if row.get('run_name') == RUN_NAME), {})

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 11,
})

print('project:', PROJECT_DIR)
print('sweep:', SWEEP_NAME)
print('run:', RUN_NAME)
print('manifest row:', {k: RUN_ROW.get(k) for k in ['arch', 'dataset_tag', 'dataset_size', 'checkpoint_dir'] if k in RUN_ROW})
print('ddpm:', DDPM_PATH, 'exists=', DDPM_PATH.exists())
print('dpm:', DPM_PATH, 'exists=', DPM_PATH.exists())
print('output:', OUT_DIR)


## Build The One-Run Comparison Table

This cell writes a small sampler-comparison table directly from the two `.npz` sample files. It does not call the older Nick-specific comparison script, so the notebook can be reused for the Fig. 2 `u64 d2p15` run.


In [ ]:
def _load_npz_for_compare(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W) or (N,H,W), got {arr.shape} from {path}')
    return arr


def _evenly_limit_for_compare(arr: np.ndarray, limit: int | None) -> np.ndarray:
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=np.int64)
    return np.array(arr[idx], copy=True)


def _summarize_pixels(arr: np.ndarray) -> dict[str, float]:
    flat = arr.reshape(arr.shape[0], -1)
    return {
        'pixel_mean': float(np.mean(flat)),
        'pixel_std': float(np.std(flat)),
        'pixel_p01': float(np.percentile(flat, 1)),
        'pixel_p50': float(np.percentile(flat, 50)),
        'pixel_p99': float(np.percentile(flat, 99)),
        'sample_mean_std': float(np.std(np.mean(flat, axis=1))),
        'sample_std_mean': float(np.mean(np.std(flat, axis=1))),
    }


def _histogram_l1_for_compare(a: np.ndarray, b: np.ndarray, bins: int = 120) -> float:
    lo = float(min(np.min(a), np.min(b)))
    hi = float(max(np.max(a), np.max(b)))
    edges = np.linspace(lo, hi, bins + 1)
    ha, _ = np.histogram(a.reshape(-1), bins=edges, density=True)
    hb, _ = np.histogram(b.reshape(-1), bins=edges, density=True)
    width = float(np.mean(np.diff(edges)))
    return float(np.sum(np.abs(ha - hb)) * width)

if not DDPM_PATH.exists() or not DPM_PATH.exists():
    raise FileNotFoundError(f'Missing sampler inputs:\nDDPM: {DDPM_PATH}\nDPM:  {DPM_PATH}')

ddpm_cmp = _evenly_limit_for_compare(_load_npz_for_compare(DDPM_PATH), MAX_SAMPLES)
dpm_cmp = _evenly_limit_for_compare(_load_npz_for_compare(DPM_PATH), MAX_SAMPLES)
n_cmp = min(len(ddpm_cmp), len(dpm_cmp))
ddpm_cmp = ddpm_cmp[:n_cmp]
dpm_cmp = dpm_cmp[:n_cmp]

ddpm_stats = _summarize_pixels(ddpm_cmp)
dpm_stats = _summarize_pixels(dpm_cmp)
row = {
    'run_name': RUN_NAME,
    'sweep_name': SWEEP_NAME,
    'arch': RUN_ROW.get('arch'),
    'dataset_tag': RUN_ROW.get('dataset_tag'),
    'dataset_size': int(RUN_ROW.get('dataset_size', -1)),
    'checkpoint_dir': RUN_ROW.get('checkpoint_dir'),
    'ddpm_label': DDPM_LABEL,
    'dpm_label': DPM_LABEL,
    'n_compared': int(n_cmp),
    'ddpm_sample_path': str(DDPM_PATH),
    'dpm_sample_path': str(DPM_PATH),
    'hist_l1': _histogram_l1_for_compare(ddpm_cmp, dpm_cmp),
    'abs_mean_delta': abs(dpm_stats['pixel_mean'] - ddpm_stats['pixel_mean']),
    'abs_std_delta': abs(dpm_stats['pixel_std'] - ddpm_stats['pixel_std']),
    'abs_p01_delta': abs(dpm_stats['pixel_p01'] - ddpm_stats['pixel_p01']),
    'abs_p99_delta': abs(dpm_stats['pixel_p99'] - ddpm_stats['pixel_p99']),
    **{f'ddpm_{k}': v for k, v in ddpm_stats.items()},
    **{f'dpm_{k}': v for k, v in dpm_stats.items()},
}
metrics_table = pd.DataFrame([row])
metrics_table.to_csv(CSV_PATH, index=False)
SUMMARY_PATH.write_text(json.dumps({
    'sweep_name': SWEEP_NAME,
    'run_name': RUN_NAME,
    'ddpm_label': DDPM_LABEL,
    'dpm_label': DPM_LABEL,
    'seed': SEED,
    'max_samples': MAX_SAMPLES,
    'n_compared': int(n_cmp),
    'csv': str(CSV_PATH),
}, indent=2) + '\n')
print('wrote', CSV_PATH)
print('wrote', SUMMARY_PATH)


In [ ]:
metrics = pd.read_csv(CSV_PATH)
display_cols = [
    'run_name', 'arch', 'dataset_tag', 'dataset_size', 'n_compared',
    'hist_l1', 'abs_mean_delta', 'abs_std_delta', 'abs_p01_delta', 'abs_p99_delta',
    'ddpm_pixel_mean', 'dpm_pixel_mean', 'ddpm_pixel_std', 'dpm_pixel_std',
]
display(metrics[[c for c in display_cols if c in metrics.columns]])
if SUMMARY_PATH.exists():
    print(SUMMARY_PATH.read_text())


## Load Samples

In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W) or (N,H,W), got {arr.shape}')
    return arr

def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=np.int64)
    return np.array(arr[idx], copy=True)

ddpm = evenly_limit(load_npz_array(DDPM_PATH), MAX_SAMPLES)
dpm = evenly_limit(load_npz_array(DPM_PATH), MAX_SAMPLES)
n = min(len(ddpm), len(dpm))
ddpm = ddpm[:n]
dpm = dpm[:n]
print('ddpm shape:', ddpm.shape, 'finite:', np.isfinite(ddpm).all())
print('dpm50 shape:', dpm.shape, 'finite:', np.isfinite(dpm).all())
print('ddpm range:', float(ddpm.min()), float(ddpm.max()))
print('dpm50 range:', float(dpm.min()), float(dpm.max()))

## One-Point Pixel PDF And Field-Level QA

The pixel histogram is the one-point statistic: it compares the marginal PDF of normalized field values after flattening all generated maps. This section reports percentile shifts, histogram L1, KS distance, smoothed KL divergences, and Jensen-Shannon divergence. KL is included, but it should not be the only criterion: empirical histogram KL is directional, binning-dependent, and becomes infinite when a bin has nonzero probability in one sample and zero in the other. Here KL is computed on a histogram with a small Jeffreys-style pseudocount (`0.5` per bin), and JS is reported as a symmetric, finite companion metric. The per-field mean/std summaries below are only QA diagnostics, not strong cosmological validation metrics.


In [ ]:
def flat_pixels(arr: np.ndarray) -> np.ndarray:
    return np.asarray(arr, dtype=np.float64).reshape(-1)


def one_point_stats(arr: np.ndarray, label: str) -> pd.DataFrame:
    x = flat_pixels(arr)
    mu = float(np.mean(x))
    sigma = float(np.std(x))
    z = (x - mu) / max(sigma, 1e-30)
    qs = [0.1, 1, 5, 16, 50, 84, 95, 99, 99.9]
    row = {
        'sampler': label,
        'n_pixels': int(x.size),
        'mean': mu,
        'std': sigma,
        'skew': float(np.mean(z ** 3)),
        'excess_kurtosis': float(np.mean(z ** 4) - 3.0),
        'min': float(np.min(x)),
        'max': float(np.max(x)),
    }
    for q in qs:
        row[f'p{str(q).replace(".", "p")}'] = float(np.percentile(x, q))
    return pd.DataFrame([row])


def one_point_distances(a: np.ndarray, b: np.ndarray, bins: int = 240, pseudocount: float = 0.5) -> pd.DataFrame:
    xa = flat_pixels(a)
    xb = flat_pixels(b)
    lo = float(min(xa.min(), xb.min()))
    hi = float(max(xa.max(), xb.max()))
    edges = np.linspace(lo, hi, bins + 1)
    ca, _ = np.histogram(xa, bins=edges)
    cb, _ = np.histogram(xb, bins=edges)
    pa = ca / max(ca.sum(), 1)
    pb = cb / max(cb.sum(), 1)

    pa_s = (ca.astype(np.float64) + pseudocount)
    pb_s = (cb.astype(np.float64) + pseudocount)
    pa_s = pa_s / pa_s.sum()
    pb_s = pb_s / pb_s.sum()
    mix = 0.5 * (pa_s + pb_s)
    kl_ab = float(np.sum(pa_s * np.log(pa_s / pb_s)))
    kl_ba = float(np.sum(pb_s * np.log(pb_s / pa_s)))
    js = float(0.5 * np.sum(pa_s * np.log(pa_s / mix)) + 0.5 * np.sum(pb_s * np.log(pb_s / mix)))

    pdf_l1 = float(np.sum(np.abs(pa - pb)))
    cdf_ks = float(np.max(np.abs(np.cumsum(pa) - np.cumsum(pb))))
    qgrid = np.linspace(0.1, 99.9, 999)
    qa = np.percentile(xa, qgrid)
    qb = np.percentile(xb, qgrid)
    return pd.DataFrame([{
        'comparison': f'{DDPM_NAME} vs {DPM_NAME}',
        'histogram_bins': int(bins),
        'kl_pseudocount': float(pseudocount),
        'histogram_L1_probability': pdf_l1,
        'cdf_KS': cdf_ks,
        'KL_DDPM_to_other_nats': kl_ab,
        'KL_other_to_DDPM_nats': kl_ba,
        'JS_divergence_nats': js,
        'quantile_MAE': float(np.mean(np.abs(qb - qa))),
        'quantile_max_abs_delta': float(np.max(np.abs(qb - qa))),
    }])


def field_summary(arr: np.ndarray, label: str) -> pd.DataFrame:
    flat = arr.reshape(len(arr), -1)
    return pd.DataFrame({
        'sampler': label,
        'field_mean': flat.mean(axis=1),
        'field_std': flat.std(axis=1),
        'field_p01': np.percentile(flat, 1, axis=1),
        'field_p99': np.percentile(flat, 99, axis=1),
    })

one_point_df = pd.concat([
    one_point_stats(ddpm, DDPM_NAME),
    one_point_stats(dpm, DPM_NAME),
], ignore_index=True)
metric_cols = [c for c in one_point_df.columns if c not in {'sampler', 'n_pixels'}]
one_point_delta = pd.DataFrame({
    'metric': metric_cols,
    DDPM_NAME: [one_point_df.loc[0, c] for c in metric_cols],
    DPM_NAME: [one_point_df.loc[1, c] for c in metric_cols],
})
one_point_delta['delta'] = one_point_delta[DPM_NAME] - one_point_delta[DDPM_NAME]
one_point_delta['abs_delta'] = one_point_delta['delta'].abs()
one_point_delta['rel_delta'] = one_point_delta['abs_delta'] / np.maximum(np.abs(one_point_delta[DDPM_NAME]), 1e-12)

one_point_distance_df = one_point_distances(ddpm, dpm)
print('one-point PDF summary')
display(one_point_delta.round(6))
print('one-point PDF distances')
display(one_point_distance_df.round(6))

onepoint_csv = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_vs_{DPM_LABEL}_one_point_stats.csv'
one_point_delta.to_csv(onepoint_csv, index=False)
print('saved', onepoint_csv)

summary_df = pd.concat([
    field_summary(ddpm, DDPM_NAME),
    field_summary(dpm, DPM_NAME),
], ignore_index=True)
print('field-level QA summaries; useful for sanity checks, not decisive scientific metrics')
display(summary_df.groupby('sampler').agg(['mean', 'std', 'min', 'max']).round(5))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
bins = np.linspace(float(min(ddpm.min(), dpm.min())), float(max(ddpm.max(), dpm.max())), 180)
axes[0].hist(ddpm.ravel(), bins=bins, density=True, histtype='step', lw=2, label=DDPM_NAME)
axes[0].hist(dpm.ravel(), bins=bins, density=True, histtype='step', lw=2, label=DPM_NAME)
axes[0].set_title('one-point pixel PDF')
axes[0].set_xlabel('normalized field value')
axes[0].set_ylabel('density')
axes[0].legend(frameon=False)

qgrid = np.linspace(0.5, 99.5, 199)
axes[1].plot(qgrid, np.percentile(flat_pixels(ddpm), qgrid), label=DDPM_NAME)
axes[1].plot(qgrid, np.percentile(flat_pixels(dpm), qgrid), label=DPM_NAME)
axes[1].set_title('one-point quantile function')
axes[1].set_xlabel('percentile')
axes[1].set_ylabel('field value')
axes[1].legend(frameon=False)

axes[2].hist(summary_df.loc[summary_df.sampler == DDPM_NAME, 'field_std'], bins=50, alpha=0.55, label=DDPM_NAME)
axes[2].hist(summary_df.loc[summary_df.sampler == DPM_NAME, 'field_std'], bins=50, alpha=0.55, label=DPM_NAME)
axes[2].set_title('field-level std QA')
axes[2].set_xlabel('per-field std')
axes[2].legend(frameon=False)

fig.tight_layout()
out = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_vs_{DPM_LABEL}_one_point_pdf.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()


## Power Spectrum Comparison

In [ ]:
PK_NBINS = 25
pk_ddpm, k = batch_power_spectra(ddpm, nbins=PK_NBINS)
pk_dpm, _ = batch_power_spectra(dpm, nbins=PK_NBINS)
pk_ddpm_mean = np.nanmean(pk_ddpm, axis=0)
pk_dpm_mean = np.nanmean(pk_dpm, axis=0)
ratio = pk_dpm_mean / np.clip(pk_ddpm_mean, 1e-30, None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].plot(k, pk_ddpm_mean, marker='o', label=DDPM_NAME)
axes[0].plot(k, pk_dpm_mean, marker='o', label=DPM_NAME)
axes[0].set_yscale('log')
axes[0].set_xlabel('k bin')
axes[0].set_ylabel('mean P(k)')
axes[0].set_title('mean power spectrum')
axes[0].legend(frameon=False)

axes[1].axhline(1.0, color='black', lw=1)
axes[1].plot(k, ratio, marker='o', color='tab:purple')
axes[1].set_xlabel('k bin')
axes[1].set_ylabel(f'{DPM_NAME} / {DDPM_NAME}')
axes[1].set_title('power ratio')
axes[1].grid(alpha=0.25)

fig.tight_layout()
out = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_vs_{DPM_LABEL}_power.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
print('pk ratio low/mid/high:', [float(np.nanmean(x)) for x in np.array_split(ratio[np.isfinite(ratio)], 3)])
plt.show()


## Image Grid

The rows are a visual marginal check, not an image-by-image pairing test. In this Fig. 2 comparison both sample files were produced by `scripts/sample_cosmodiff.py` with the same seed, batch size, model checkpoint, and device, so the initial Gaussian noise stream should be aligned by index. The final fields can still differ because DDPM500 and DPM-Solver50 follow different reverse-time update rules, and DDPM also injects stochastic noise during reverse steps.


In [ ]:
N_SHOW = 8
idx = np.linspace(0, n - 1, N_SHOW, dtype=int)
vals = np.concatenate([ddpm[idx, 0].ravel(), dpm[idx, 0].ravel()])
vmin, vmax = np.nanpercentile(vals, [1, 99])

fig, axes = plt.subplots(2, N_SHOW, figsize=(2.1 * N_SHOW, 4.4), constrained_layout=True)
for col, i in enumerate(idx):
    axes[0, col].imshow(ddpm[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0, col].set_title(f'{DDPM_NAME} {i}', fontsize=10)
    axes[1, col].imshow(dpm[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1, col].set_title(f'{DPM_NAME} {i}', fontsize=10)
    for ax in axes[:, col]:
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle(f'{RUN_NAME}: {DDPM_NAME} vs {DPM_NAME}; same index is not a matched physical sample', y=1.02)
out = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_vs_{DPM_LABEL}_image_grid.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()


## Nearest-Neighbor Visual Matching

The same-index grid above is intentionally conservative: it shows that sampler trajectories are not paired images. In this notebook both files come from the same wrapper and checkpoint, so index alignment is better controlled than the old Nick `raw_train_full` versus `dpm50` comparison. Even then, DDPM injects reverse-step noise and DPM-Solver follows a different solver trajectory, so the final images need not match one-to-one.

A more useful visual check is therefore nearest-neighbor matching: for a few DPM samples, find the closest DDPM sample in normalized image space. Here “nearest” uses per-image standardized pixel cosine similarity. This is only a morphology/visual proxy; it is not a cosmological metric and it is sensitive to translations/rotations. Use it to decide whether the samplers produce visually comparable fields, then rely on one-point PDFs, power spectra, and PCA/SSCD diagnostics for scientific comparison.


In [ ]:
def standardized_flat_images(arr: np.ndarray) -> np.ndarray:
    flat = np.asarray(arr[:, 0], dtype=np.float32).reshape(len(arr), -1)
    flat = flat - flat.mean(axis=1, keepdims=True)
    flat = flat / np.maximum(flat.std(axis=1, keepdims=True), 1e-6)
    flat = flat / np.maximum(np.linalg.norm(flat, axis=1, keepdims=True), 1e-12)
    return flat


def nearest_by_cosine(query: np.ndarray, reference: np.ndarray, query_indices: np.ndarray) -> pd.DataFrame:
    q = standardized_flat_images(query)
    r = standardized_flat_images(reference)
    sims = q[query_indices] @ r.T
    nn_idx = np.argmax(sims, axis=1)
    rows = []
    for row, q_idx in enumerate(query_indices):
        ref_idx = int(nn_idx[row])
        diff = query[q_idx, 0] - reference[ref_idx, 0]
        rows.append({
            'query_sampler': DPM_NAME,
            'query_index': int(q_idx),
            'nearest_sampler': DDPM_NAME,
            'nearest_index': ref_idx,
            'standardized_pixel_cosine': float(sims[row, ref_idx]),
            'pixel_mse': float(np.mean(diff ** 2)),
            'query_field_std': float(query[q_idx, 0].std()),
            'nearest_field_std': float(reference[ref_idx, 0].std()),
        })
    return pd.DataFrame(rows)

N_MATCH = 6
query_idx = np.linspace(0, n - 1, N_MATCH, dtype=int)
nearest_df = nearest_by_cosine(dpm, ddpm, query_idx)
display(nearest_df.round(5))

nearest_csv = OUT_DIR / f'{RUN_NAME}_{DPM_LABEL}_nearest_{DDPM_LABEL}_visual_matches.csv'
nearest_df.to_csv(nearest_csv, index=False)
print('saved', nearest_csv)

matched_vals = []
for row in nearest_df.itertuples(index=False):
    matched_vals.append(dpm[int(row.query_index), 0].ravel())
    matched_vals.append(ddpm[int(row.nearest_index), 0].ravel())
vmin, vmax = np.nanpercentile(np.concatenate(matched_vals), [1, 99])

fig, axes = plt.subplots(3, N_MATCH, figsize=(2.25 * N_MATCH, 6.2), constrained_layout=True)
for col, row in enumerate(nearest_df.itertuples(index=False)):
    q_idx = int(row.query_index)
    ref_idx = int(row.nearest_index)
    query_img = dpm[q_idx, 0]
    ref_img = ddpm[ref_idx, 0]
    diff_img = query_img - ref_img
    axes[0, col].imshow(query_img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0, col].set_title(f'{DPM_NAME} {q_idx}', fontsize=9)
    axes[1, col].imshow(ref_img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1, col].set_title(f'NN {DDPM_NAME} {ref_idx}\ncos={row.standardized_pixel_cosine:.3f}', fontsize=9)
    dlim = np.nanpercentile(np.abs(diff_img), 99)
    axes[2, col].imshow(diff_img, origin='lower', cmap='coolwarm', vmin=-dlim, vmax=dlim)
    axes[2, col].set_title('difference', fontsize=9)
    for ax in axes[:, col]:
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle(f'{RUN_NAME}: nearest {DDPM_NAME} image for selected {DPM_NAME} samples', y=1.02)
out = OUT_DIR / f'{RUN_NAME}_{DPM_LABEL}_nearest_{DDPM_LABEL}_image_grid.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()


## Generated-To-Training Memorization Check

The DPM-vs-DDPM nearest-neighbor panel is not a memorization test. It compares two independent sampler outputs from the same model, so low image-level similarity there only says the samplers did not land on the same individual field. To test the concern that the model is memorizing, compare generated fields directly against the normalized training fields used for this run.

This section loads the training slices from the manifest/config, applies the same training normalization, and finds the nearest training field for selected generated samples using per-image standardized pixel cosine similarity. If the model is copying training fields, the nearest training image should look almost identical, the cosine should be close to 1, and the difference map should be close to zero. Matching one-point PDFs or P(k) alone is weaker: those are ensemble summaries and do not require phase-level/image-level identity.


In [ ]:
import yaml

TRAIN_NN_SHOW = int(os.environ.get('NF_TRAIN_NN_SHOW', '6'))
TRAIN_NN_REFERENCE_LIMIT_RAW = int(os.environ.get('NF_TRAIN_NN_REFERENCE_LIMIT', '0'))
TRAIN_NN_REFERENCE_LIMIT = TRAIN_NN_REFERENCE_LIMIT_RAW if TRAIN_NN_REFERENCE_LIMIT_RAW > 0 else None
TRAIN_NN_REF_BATCH = int(os.environ.get('NF_TRAIN_NN_REF_BATCH', '2048'))


def _as_nchw_train(images: np.ndarray) -> np.ndarray:
    arr = np.asarray(images)
    if arr.ndim == 3:
        return arr[:, None, :, :]
    if arr.ndim == 4 and arr.shape[1] == 1:
        return arr
    raise ValueError(f'Expected (N,H,W) or (N,1,H,W), got {arr.shape}')


def _load_source_slices_with_meta(source: dict, zthin: int) -> tuple[np.ndarray, list[dict]]:
    path = Path(source['path'])
    n_train = int(source['n_samples'])
    tag = source.get('tag', path.stem)
    arr = np.load(path, mmap_mode='r')
    stop = min(n_train, len(arr))
    z_indices = np.arange(0, arr.shape[1], zthin, dtype=np.int64)
    slices = np.asarray(arr[:stop, ::zthin], dtype=np.float32)
    images = slices.reshape(-1, slices.shape[-2], slices.shape[-1])
    meta = []
    for sim_index in range(stop):
        for z_index in z_indices:
            meta.append({
                'source_tag': tag,
                'source_path': str(path),
                'source_sim_index': int(sim_index),
                'source_slice_index': int(z_index),
            })
    return images, meta


def _tanh_transform(images: np.ndarray, alpha: float, beta: float, gamma: float, delta: float, sigma: float, mu: float) -> np.ndarray:
    shifted = images - np.float32(mu)
    pos = alpha * np.tanh((gamma * shifted) / alpha)
    neg = beta * np.tanh((delta * shifted) / beta)
    return (np.where(shifted >= 0, pos, neg) * sigma).astype(np.float32, copy=False)


def _normalize_like_training(train_images: np.ndarray, config: dict) -> tuple[np.ndarray, dict[str, float]]:
    data_cfg = config['data']
    transform = data_cfg.get('transform', None)
    use_log = bool(data_cfg.get('log', False)) or (
        isinstance(transform, (list, tuple)) and 'log' in transform
    )
    train = train_images.astype(np.float32, copy=False)
    if use_log:
        np.log(train, out=train)

    normalization = data_cfg.get('normalization', None)
    norm_kwargs = dict(data_cfg.get('norm_kwargs') or {})
    info = {'use_log': float(use_log)}

    if normalization in {'tanh', 'centermax', 'center-max', 'centered_maxabs'}:
        center = norm_kwargs.get('center', None)
        if center is None:
            center = float(train.mean())
        train -= np.float32(center)

        xmax = norm_kwargs.get('xmax', None)
        if xmax is None:
            xmax = float(np.abs(train).max())
        xmax = max(float(xmax), 1e-30)
        train /= np.float32(xmax)
        info.update({'center': float(center), 'xmax': float(xmax)})

    if normalization == 'tanh':
        alpha = float(norm_kwargs.get('alpha', 1.0))
        beta = float(norm_kwargs.get('beta', 1.0))
        gamma = float(norm_kwargs.get('gamma', 1.0))
        delta = float(norm_kwargs.get('delta', 1.0))
        sigma = float(norm_kwargs.get('sigma', 1.0))
        mu = float(norm_kwargs.get('mu', 0.0))
        train = _tanh_transform(train, alpha, beta, gamma, delta, sigma, mu)
    elif normalization in {None, 'none', 'centermax', 'center-max', 'centered_maxabs'}:
        pass
    else:
        raise ValueError(f'Unsupported normalization for this notebook: {normalization!r}')

    info.update({'normalization': str(normalization), 'n_train_loaded': float(len(train))})
    return _as_nchw_train(train), info


def load_training_reference_for_run() -> tuple[np.ndarray, pd.DataFrame, dict[str, float]]:
    if not RUN_ROW or 'source_counts' not in RUN_ROW:
        raise RuntimeError('Manifest row with source_counts is required for generated-to-training NN checks.')
    config_path = PROJECT_DIR / RUN_ROW['config']
    with config_path.open() as f:
        config = yaml.safe_load(f)
    zthin = int(RUN_ROW.get('zthin', config['data'].get('zthin', 1)))

    image_parts = []
    meta_rows = []
    for source in RUN_ROW['source_counts']:
        images_part, meta_part = _load_source_slices_with_meta(source, zthin)
        image_parts.append(images_part)
        meta_rows.extend(meta_part)
    train_raw = np.concatenate(image_parts, axis=0)
    train_ref, norm_info = _normalize_like_training(train_raw, config)

    train_meta = pd.DataFrame(meta_rows)
    train_meta.insert(0, 'training_index', np.arange(len(train_meta), dtype=np.int64))
    if len(train_meta) != len(train_ref):
        raise RuntimeError(f'Training metadata/image mismatch: {len(train_meta)} vs {len(train_ref)}')

    if TRAIN_NN_REFERENCE_LIMIT is not None and len(train_ref) > TRAIN_NN_REFERENCE_LIMIT:
        keep = np.linspace(0, len(train_ref) - 1, TRAIN_NN_REFERENCE_LIMIT, dtype=np.int64)
        train_ref = train_ref[keep]
        train_meta = train_meta.iloc[keep].reset_index(drop=True)
        norm_info['reference_limit'] = float(TRAIN_NN_REFERENCE_LIMIT)
    else:
        norm_info['reference_limit'] = float(len(train_ref))
    return train_ref, train_meta, norm_info


def nearest_training_by_cosine(query: np.ndarray, query_label: str, reference: np.ndarray, reference_meta: pd.DataFrame, query_indices: np.ndarray) -> pd.DataFrame:
    q = standardized_flat_images(query[query_indices])
    best_sim = np.full(len(query_indices), -np.inf, dtype=np.float32)
    best_loaded = np.full(len(query_indices), -1, dtype=np.int64)

    for start in range(0, len(reference), TRAIN_NN_REF_BATCH):
        stop = min(start + TRAIN_NN_REF_BATCH, len(reference))
        r = standardized_flat_images(reference[start:stop])
        sims = q @ r.T
        local = np.argmax(sims, axis=1)
        local_sim = sims[np.arange(len(query_indices)), local]
        improve = local_sim > best_sim
        best_sim[improve] = local_sim[improve]
        best_loaded[improve] = start + local[improve]

    rows = []
    for i, q_idx in enumerate(query_indices):
        loaded_idx = int(best_loaded[i])
        train_row = reference_meta.iloc[loaded_idx].to_dict()
        diff = query[int(q_idx), 0] - reference[loaded_idx, 0]
        rows.append({
            'query_sampler': query_label,
            'query_index': int(q_idx),
            'nearest_set': 'training',
            'nearest_loaded_index': loaded_idx,
            'nearest_training_index': int(train_row['training_index']),
            'nearest_source_tag': train_row.get('source_tag'),
            'nearest_source_sim_index': int(train_row.get('source_sim_index')),
            'nearest_source_slice_index': int(train_row.get('source_slice_index')),
            'standardized_pixel_cosine': float(best_sim[i]),
            'pixel_mse': float(np.mean(diff ** 2)),
            'query_field_mean': float(query[int(q_idx), 0].mean()),
            'query_field_std': float(query[int(q_idx), 0].std()),
            'train_field_mean': float(reference[loaded_idx, 0].mean()),
            'train_field_std': float(reference[loaded_idx, 0].std()),
        })
    return pd.DataFrame(rows)


def plot_generated_to_train_nn(samples: np.ndarray, sample_name: str, sample_label: str, nn_df: pd.DataFrame, reference: np.ndarray) -> Path:
    vals = []
    for row in nn_df.itertuples(index=False):
        vals.append(samples[int(row.query_index), 0].ravel())
        vals.append(reference[int(row.nearest_loaded_index), 0].ravel())
    vmin, vmax = np.nanpercentile(np.concatenate(vals), [1, 99])

    fig, axes = plt.subplots(3, len(nn_df), figsize=(2.25 * len(nn_df), 6.2), constrained_layout=True)
    if len(nn_df) == 1:
        axes = axes[:, None]
    for col, row in enumerate(nn_df.itertuples(index=False)):
        q_idx = int(row.query_index)
        train_idx = int(row.nearest_loaded_index)
        query_img = samples[q_idx, 0]
        train_img = reference[train_idx, 0]
        diff_img = query_img - train_img
        axes[0, col].imshow(query_img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[0, col].set_title(f'{sample_name} {q_idx}', fontsize=9)
        axes[1, col].imshow(train_img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].set_title(
            f"NN train {int(row.nearest_training_index)}\ncos={row.standardized_pixel_cosine:.3f}",
            fontsize=9,
        )
        dlim = np.nanpercentile(np.abs(diff_img), 99)
        dlim = max(float(dlim), 1e-6)
        axes[2, col].imshow(diff_img, origin='lower', cmap='coolwarm', vmin=-dlim, vmax=dlim)
        axes[2, col].set_title('generated - train', fontsize=9)
        for ax in axes[:, col]:
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle(f'{RUN_NAME}: nearest training field for selected {sample_name} samples', y=1.02)
    out = OUT_DIR / f'{RUN_NAME}_{sample_label}_nearest_train_image_grid.png'
    fig.savefig(out, bbox_inches='tight')
    print('saved', out)
    plt.show()
    return out


train_ref, train_meta, train_norm_info = load_training_reference_for_run()
print('loaded training reference:', train_ref.shape)
print('training normalization:', train_norm_info)
print('reference metadata head:')
display(train_meta.head())

train_query_idx = np.linspace(0, n - 1, min(TRAIN_NN_SHOW, n), dtype=int)
dpm_train_nn = nearest_training_by_cosine(dpm, DPM_NAME, train_ref, train_meta, train_query_idx)
ddpm_train_nn = nearest_training_by_cosine(ddpm, DDPM_NAME, train_ref, train_meta, train_query_idx)
train_nn_df = pd.concat([dpm_train_nn, ddpm_train_nn], ignore_index=True)
display(train_nn_df.round(5))

train_nn_csv = OUT_DIR / f'{RUN_NAME}_{DDPM_LABEL}_{DPM_LABEL}_nearest_train_visual_matches.csv'
train_nn_df.to_csv(train_nn_csv, index=False)
print('saved', train_nn_csv)

plot_generated_to_train_nn(dpm, DPM_NAME, DPM_LABEL, dpm_train_nn, train_ref)
plot_generated_to_train_nn(ddpm, DDPM_NAME, DDPM_LABEL, ddpm_train_nn, train_ref)


## Files Written

In [ ]:
patterns = [
    f'*{DPM_LABEL}*',
    f'*{DDPM_LABEL}_vs_{DPM_LABEL}*',
    f'*{DPM_LABEL}_nearest_{DDPM_LABEL}*',
    f'*nearest_train*',
]
seen = set()
for pattern in patterns:
    for path in sorted(OUT_DIR.glob(pattern)):
        if path in seen:
            continue
        seen.add(path)
        print(path, path.stat().st_size)
